# Laser Marking with Sketches (Inline)

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.

The sketch is solved and marked straight from the request: nothing is stored in the datastore beforehand. Once the run is over, its log is requested from the datastore. To keep the sketch in the database and mark it from there, see `laser_marking.ipynb`.

In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api import DatastoreApi, InstrumentApi

datastore_host = "localhost:5208"
instrument_host = "localhost:5210"

with DatastoreApi(host=datastore_host) as datastore_api:
    datastore_health_response = datastore_api.health.GetHealth(HealthRequest())
    print("Datastore Health:")
    print(datastore_health_response)

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_health_response = instrument_api.health.GetHealth(HealthRequest())
    print("Instrument Health:")
    print(instrument_health_response)

## Build a Sketch

The sketch below holds four geometries, all in microns:

- `dot`: an explicit point marked with a single pulse.
- `waveguide`: an explicit line string without marking parameters. It is never marked and only serves as a reference.
- `cut`: a line string computed as the offset of `waveguide`, 10 µm to its right, marked with a pulse train.
- `pad_fill`: an explicit polygon hatched with a cross pattern.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import (
    Coordinate,
    GeometryDistanceUnit,
    LineString,
    Point,
    Polygon,
    Shape,
)
from femtum_sdk.api.contracts.v1.dto.hatching_pb2 import (
    HatchBidirectional,
    HatchContour,
    HatchContourAfterFill,
    HatchCross,
    HatchingParameters,
    HatchOrder,
    HatchPattern,
    HatchShape,
)
from femtum_sdk.api.contracts.v1.dto.marking_parameters_pb2 import (
    LaserMarkingParameters,
    MarkingParameters,
    PulseFrequency,
    PulseFrequencyMaximum,
    PulseLength,
    PulseLengthHalfPeriod,
    PulseParameters,
    SinglePulse,
    TrainPulse,
)
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import (
    AngleMeasure,
    Degrees,
    DistanceMeasure,
    EnergyMeasure,
    Nanojoules,
    Microns,
)
from femtum_sdk.api.contracts.v1.dto.sketch_pb2 import (
    ConstrainedShape,
    OffsetConstraint,
    Sketch,
    SketchConstraint,
    SketchGeometry,
    SketchLineString,
)

microns = GeometryDistanceUnit(distance=Microns)

# At the maximum pulse frequency the laser power is energy x frequency, and it must fall inside the range of the laser energy calibration.
pulse_energy = EnergyMeasure(value=100, unit=Nanojoules)

single_pulse_marking = MarkingParameters(
    laser=LaserMarkingParameters(
        laser_energy=pulse_energy,
        pulse=PulseParameters(single_pulse=SinglePulse()),
    )
)

pulse_train_marking = MarkingParameters(
    laser=LaserMarkingParameters(
        laser_energy=pulse_energy,
        pulse=PulseParameters(
            pulse_train=TrainPulse(
                frequency=PulseFrequency(maximum=PulseFrequencyMaximum()),
                length=PulseLength(half_period=PulseLengthHalfPeriod()),
            )
        ),
    )
)

sketch = Sketch(
    geometries=[
        SketchGeometry(
            id="dot",
            name="Single pulse",
            shape=Shape(point=Point(x=-50, y=0, unit=microns)),
            marking_parameters=single_pulse_marking,
        ),
        SketchGeometry(
            id="waveguide",
            name="Waveguide",
            shape=Shape(
                line_string=LineString(
                    unit=microns,
                    path=[Coordinate(x=0, y=0), Coordinate(x=200, y=0)],
                )
            ),
        ),
        SketchGeometry(
            id="cut",
            name="Cut beside the waveguide",
            constrained=ConstrainedShape(
                line_string=SketchLineString(),
                constraints=[
                    SketchConstraint(
                        id="10um_right_of_waveguide",
                        offset=OffsetConstraint(
                            reference_id="waveguide",
                            distance=DistanceMeasure(value=10, unit=Microns),
                        ),
                    )
                ],
            ),
            marking_parameters=pulse_train_marking,
        ),
        SketchGeometry(
            id="pad_fill",
            name="Pad",
            hatch=HatchShape(
                polygon=Polygon(
                    path=LineString(
                        unit=microns,
                        path=[
                            Coordinate(x=0, y=50),
                            Coordinate(x=100, y=50),
                            Coordinate(x=100, y=100),
                            Coordinate(x=0, y=100),
                            Coordinate(x=0, y=50),
                        ],
                    )
                ),
                parameters=HatchingParameters(
                    pattern=HatchPattern(
                        cross=HatchCross(
                            spacing=DistanceMeasure(value=5, unit=Microns),
                            angle=AngleMeasure(value=45, unit=Degrees),
                            padding=DistanceMeasure(value=2, unit=Microns),
                        )
                    ),
                    order=HatchOrder(bidirectional=HatchBidirectional()),
                    contour=HatchContour(after_fill=HatchContourAfterFill()),
                ),
            ),
            marking_parameters=pulse_train_marking,
        ),
    ]
)

## Solve the Sketch

Solving turns the sketch into the marking geometries of an execution plan. It never fires the laser and needs no calibration, so it is safe to call on every change.

Only geometries carrying marking parameters come back, in sketch order: `waveguide` is absent, `cut` is placed 10 µm from it, and the hatch comes back as one geometry per shape of its fill. A sketch the solver refuses is reported as an invalid argument naming the geometries or constraints at fault.

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.sketch_solver_pb2 import SolveSketchRequest, SolveSketchResponse

with InstrumentApi(host=instrument_host) as instrument_api:
    solved: SolveSketchResponse = instrument_api.sketch_solver.Solve(SolveSketchRequest(sketch=sketch))

print(f"{len(solved.marking_geometries)} marking geometries")
for marking_geometry in solved.marking_geometries[:3]:
    print(marking_geometry.shape)

## Rehearse the Plan

`Plan` runs every preparation step of a real run (energy to power conversion, calibrations) and reports what would happen, **without firing the laser**. A rehearsal leaves no log in the datastore.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.execution_plan_pb2 import ExecutionPlan
from femtum_sdk.api.contracts.v1.instruments.laser_execution_pb2 import LaserExecutionFeatures
from femtum_sdk.api.contracts.v1.instruments.plan_execution_pb2 import (
    ExecutionPlanSelector,
    LaserEnergyCalibrationLatest,
    LaserEnergyCalibrationSelector,
    PlanExecutionOptions,
    PlanExecutionRequest,
    PlanExecutionResponse,
)

execution_plan = ExecutionPlan(
    name="Inline Laser Marking Example",
    marking_geometries=solved.marking_geometries,
)
plan_options = PlanExecutionOptions(
    laser_energy_calibration_selector=LaserEnergyCalibrationSelector(latest=LaserEnergyCalibrationLatest()),
    features=LaserExecutionFeatures(take_pictures=True),
)

with InstrumentApi(host=instrument_host) as instrument_api:
    rehearsal: PlanExecutionResponse = instrument_api.plan_execution.Plan(
        PlanExecutionRequest(plan=ExecutionPlanSelector(plan=execution_plan), options=plan_options)
    )

print(rehearsal.log)

## Execute the Plan Inline

> **Warning:** this cell **fires the laser** on the component in the machine. Rehearse the plan first and make sure the machine is ready.

The laser is a safety switch that no other operation flips for you, and a run is refused while it is off, so the cell enables it first. The plan is sent with the request and is kept only as part of the log of the run.

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.femtosecond_laser_pb2 import EnableLaserRequest, IsEnabledRequest
from femtum_sdk.api.contracts.v1.instruments.plan_execution_pb2 import ExecutePlanRequest, ExecutePlanResponse

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_api.femtosecond_laser.Enable(EnableLaserRequest(enable=True))
    assert instrument_api.femtosecond_laser.IsEnabled(IsEnabledRequest()).enabled

    execution: ExecutePlanResponse = instrument_api.plan_execution.Execute(
        ExecutePlanRequest(plan=ExecutionPlanSelector(plan=execution_plan), options=plan_options)
    )

print(execution.log)

## Request the Execution Log

The response of the run carries the identifier of its log. Only a real run is logged, so this works with the identifier returned by `Execute`. Look it up in the datastore to read what was marked, the calibrations in force and the pictures attached to the run. The pictures taken before and after the marking are downloaded and shown below.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.laser_execution_pb2 import LaserExecutionLog
from femtum_sdk.api.contracts.v1.storage.laser_execution_log_pb2 import FindLaserExecutionLogByIdRequest
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import EnergyMeasureUnit

with DatastoreApi(host=datastore_host) as datastore_api:
    execution_log: LaserExecutionLog = datastore_api.laser_execution_log.FindById(
        FindLaserExecutionLogByIdRequest(id=execution.log.id)
    )
    attachments_by_name = {
        attachment_index.name: datastore_api.attachment.Download(id=attachment_index.id)
        for attachment_index in execution_log.attachments
    }

print(f"Log ID: {execution_log.id}")
print(f"Laser energy calibration ID: {execution_log.laser_energy_calibration_id}")
print(f"Galvo calibration ID: {execution_log.galvo_calibration_id}")
print(f"{len(execution_log.marked_geometries)} marked geometries")

for index, logged in enumerate(execution_log.marked_geometries):
    laser_energy = logged.marking_geometry.marking_parameters.laser.laser_energy
    shape_type = logged.marking_geometry.shape.WhichOneof("shape_type")
    print(f"#{index} {shape_type}: {laser_energy.value} {EnergyMeasureUnit.Name(laser_energy.unit)} per pulse")

## Render the Logged Geometries on the Pictures

The log stores every marked geometry in microns from the center of the context, the area the pictures show. The context size and scale of the run are recorded in the attributes of its geometries, and they place each geometry on the picture. The picture axes are those of the image: `x` to the right, `y` downwards.

In [ ]:
import io

from IPython.display import display, Markdown
from PIL import Image, ImageDraw

MICRONS_PER_MILLIMETER = 1000
POINT_RADIUS_PX = 8
LINE_WIDTH_PX = 3

# The shots of a run carry the context it was fired in; the marked geometries do not.
context = {attribute.key: attribute.value for attribute in execution_log.geometries[0].attributes}
context_width_px = float(context["context.width"])
context_height_px = float(context["context.height"])
microns_per_context_px = float(context["context.scaling_factor_mm_per_px"]) * MICRONS_PER_MILLIMETER


def vertices_in_microns(shape):
    shape_type = shape.WhichOneof("shape_type")
    if shape_type == "point":
        return [(shape.point.x, shape.point.y)]
    if shape_type == "line_string":
        return [(coordinate.x, coordinate.y) for coordinate in shape.line_string.path]
    if shape_type == "polygon":
        return [(coordinate.x, coordinate.y) for coordinate in shape.polygon.path.path]
    raise ValueError(f"Cannot draw a {shape_type}")


def draw_marked_geometries(picture_png: bytes) -> Image.Image:
    image = Image.open(io.BytesIO(picture_png)).convert("RGB")
    pixels_per_micron_x = image.width / context_width_px / microns_per_context_px
    pixels_per_micron_y = image.height / context_height_px / microns_per_context_px
    draw = ImageDraw.Draw(image)
    for logged in execution_log.marked_geometries:
        vertices_px = [
            (image.width / 2 + x * pixels_per_micron_x, image.height / 2 + y * pixels_per_micron_y)
            for x, y in vertices_in_microns(logged.marking_geometry.shape)
        ]
        if len(vertices_px) == 1:
            x, y = vertices_px[0]
            draw.ellipse((x - POINT_RADIUS_PX, y - POINT_RADIUS_PX, x + POINT_RADIUS_PX, y + POINT_RADIUS_PX), outline="red", width=LINE_WIDTH_PX)
        else:
            draw.line(vertices_px, fill="red", width=LINE_WIDTH_PX)
    return image


for title, name in (("Before", "pre_execution_picture"), ("After", "post_execution_picture")):
    display(Markdown(f"**{title}**"))
    display(draw_marked_geometries(attachments_by_name[name].blob_data))